AI and MCP notes: 

super simple sreamlit AI

In [ ]:
"""
import streamlit as st
from mcp import Client  # hypothetical MCP client

# Connect to MCP model
client = Client("http://localhost:8000")

st.title("Streamlit + MCP Example")

user_input = st.text_input("Ask the model:")

if user_input:
    response = client.query({"prompt": user_input})
    st.write("Model says:", response["output"])
"""

an MCP server

In [ ]:
from mcp.server.fastmcp import FastMCP


# Create an MCP server
mcp = FastMCP("Weather Service")

# Tool implementation
@mcp.tool()
def get_weather(location: str) -> str:
    """Get the current weather for a specified location."""
    return f"Weather in {location}: Sunny, 72°F"

# Resource implementation
@mcp.resource("weather://{location}")
def weather_resource(location: str) -> str:
    """Provide weather data as a resource."""
    return f"Weather data for {location}: Sunny, 72°F"

# Prompt implementation 
@mcp.prompt()
def weather_report(location: str) -> str:
    """Create a weather report prompt."""
    return f"""You are a weather reporter. Weather report for {location}?"""


# Run the server
if __name__ == "__main__":
    mcp.run()

a sample Copilot found on Git

In [ ]:
#from    https://github.com/Clffordojuka/MCP-client/blob/main/app.py    
      
from dotenv import load_dotenv
from openai import OpenAI
import streamlit as st
import io
import os

load_dotenv()

# Load environment variables from .env file
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
HF_API_KEY = os.getenv("HF_API_KEY")

# Check if the keys are loaded correctly
if not OPENAI_API_KEY or not HF_API_KEY:
    raise ValueError("Missing API keys")
# Example usage of the API keys
def use_openai_api():
    print(f"Using OpenAI API with key: {OPENAI_API_KEY}")

def use_huggingface_api():
    print(f"Using Hugging Face API with key: {HF_API_KEY}")

# streamlit setup
def initialize_page():
    """Initialize the Streamlit page configuration and layout"""
    st.set_page_config(
        page_title="Streamlit MCP Client",
        page_icon="🤖",
        layout="centered",
        initial_sidebar_state="auto"
    )
    st.markdown(
        """
        <style>
        .main {background-color: #f5f6fa;}
        .stButton>button {background-color: #4F8BF9; color: white;}
        .stTextArea textarea {background-color: #f0f4ff;}
        </style>
        """,
        unsafe_allow_html=True
    )
    st.title("🤖 Streamlit MCP Client")
    st.caption("Ask about any topic and get resources or summaries from your selected MCP server.")

    # Add a horizontal line for separation
    st.markdown("---")

    # Return a column object which can be used to place widgets
    return st.columns(1)[0]

def get_user_input(column):
    """Handle transcript input methods and return the transcript text"""

    with column:
        st.subheader("Enter Your Topic")
        user_text = st.text_area(
            "What are you interested in?",
            height=100,
            placeholder="E.g., Python web frameworks, quantum computing, etc.",
            help="Type your topic of interest here."
        )
    return user_text

def create_mcp_server_dropdown():
    mcp_servers = {
        "deepwiki": "https://mcp.deepwiki.com/mcp",
        "huggingface": "https://huggingface.co/mcp"
    }
    st.subheader("Choose MCP Server")
    selected_server = st.radio(
        "",
        options=list(mcp_servers.keys()),
        format_func=lambda x: "DeepWiki" if x == "deepwiki" else "HuggingFace",
        horizontal=True,
        help="Choose the MCP server you want to connect to"
    )
    server_url = mcp_servers[selected_server]
    st.caption(f"Selected server: `{server_url}`")
    return selected_server, server_url

#generate the transcript
def generate_response(user_text, selected_server, server_url):
    """Generate response using OpenAI client and MCP tools"""
    client = OpenAI() 

    try:
        mcp_tool = {
            "type": "mcp",
            "server_label": selected_server, 
            "server_url": server_url,      
            "require_approval": "never",   
        }

        if selected_server == 'huggingface':
            if HF_API_KEY:
                mcp_tool["headers"] = {"Authorization": f"Bearer {HF_API_KEY}"}
            else:
                st.warning("Hugging Face API Key not found in .env. Some functionalities might be limited.")
            prompt_text = f"List some resources relevant to this topic: {user_text}?"
        else:
            prompt_text = f"Summarize codebase contents relevant to this topic: {user_text}?"

        response = client.chat.completions.create(
            model="gpt-3.5-turbo", 
            tools=[mcp_tool],      
            messages=[
                {"role": "user", "content": prompt_text}
            ]
        )

        st.info(
            f"""
            **Response:**
            {response.output_text}
            """
        )
        return response

    except Exception as e:
        st.error(f"Error generating response: {str(e)}") 
        return None
    
# Main function to run the Streamlit app
def main():
    # 1. Initialize the page layout
    main_column = initialize_page()

    # 2. Get user input for the topic
    user_text = get_user_input(main_column)

    # 3. Allow user to select the MCP server
    with main_column: # Place the radio buttons within the main column
        selected_server, server_url = create_mcp_server_dropdown()

    # 4. Add a button to trigger the response generation
    if st.button("Generate Response", key="generate_button"):
        if user_text:
            with st.spinner("Generating response..."): 
                generate_response(user_text, selected_server, server_url)
        else:
            st.warning("Please enter a topic first.")

if __name__ == "__main__":
    main()